# Shared pipeline setup

Common boilerplate for all update-* pipeline notebooks.  
Loads packages, sets up environment variables, connects to the pins board,
and provides shared helper functions.

**Usage** — add this as the first code cell of every pipeline notebook:

```r
# %run ./_shared_setup
```

In [0]:
%r
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/latest"))

# Common packages used by all pipeline notebooks
common_pkgs <- c("pins", "arrow", "dplyr", "glue")
to_install <- common_pkgs[!vapply(common_pkgs, requireNamespace, logical(1), quietly = TRUE)]
if (length(to_install) > 0) install.packages(to_install)

# Safely attach packages — accept the already-loaded version if a
# newer on-disk copy cannot replace it (e.g. dplyr loaded by sparklyr)
safe_library <- function(pkgs) {
  invisible(lapply(pkgs, function(p) {
    tryCatch(
      library(p, character.only = TRUE),
      error = function(e) {
        if (isNamespaceLoaded(p)) {
          message("Using pre-loaded ", p, " ", packageVersion(p))
        } else {
          stop(e)
        }
      }
    )
  }))
}

safe_library(common_pkgs)

In [0]:
%r
# Set required env vars from the notebook context
Sys.setenv(DATABRICKS_HOST = paste0("https://", as.character(SparkR::sparkR.conf("spark.databricks.workspaceUrl"))))

nb_ctx <- SparkR::sparkR.callJMethod(
  SparkR::sparkR.callJStatic("com.databricks.dbutils_v1.DBUtilsHolder", "dbutils"),
  "notebook"
)
ctx    <- SparkR::sparkR.callJMethod(nb_ctx, "getContext")
token  <- SparkR::sparkR.callJMethod(SparkR::sparkR.callJMethod(ctx, "apiToken"), "get")
Sys.setenv(DATABRICKS_TOKEN = token)

In [0]:
%r
VOLUME_PATH <- "/Volumes/catalog_40_copper_statistics_services/postcode_mp/pins"

board <- board_databricks(
  folder_url = VOLUME_PATH,
  versioned  = TRUE
)

#' Read a single metadata field from an existing pin.
#' Returns NULL if the pin does not exist yet.
read_pin_meta_field <- function(board, pin_name, field) {
  tryCatch({
    m <- pin_meta(board, pin_name)
    m$user[[field]]
  }, error = function(e) NULL)
}

#' Validate a data frame, write it to parquet, and upload as a pin.
#'
#' Wrapping this in a function means on.exit() reliably cleans up the
#' temp file, and validation + error handling are consistent everywhere.
pin_parquet <- function(board, df, pin_name, title, description, metadata,
                        expected_cols = NULL) {
  # --- validation ---
  stopifnot(
    "Data frame is empty" = nrow(df) > 0
  )
  if (!is.null(expected_cols)) {
    missing <- setdiff(expected_cols, names(df))
    if (length(missing) > 0) {
      stop("Missing expected columns: ", paste(missing, collapse = ", "))
    }
  }

  # --- write to temp parquet ---
  tmp <- tempfile(fileext = ".parquet")
  on.exit(unlink(tmp), add = TRUE)
  arrow::write_parquet(df, tmp, compression = "zstd")

  # --- upload pin with error handling ---
  tryCatch(
    board |> pin_upload(
      paths       = tmp,
      name        = pin_name,
      title       = title,
      description = description,
      metadata    = metadata
    ),
    error = function(e) {
      stop("pin_upload failed for '", pin_name, "': ", conditionMessage(e))
    }
  )
}